# Discourse markers and speaker gender in the Spoken BNC2014 — v2 experiments

This notebook reruns every experiment for the paper with the fixes from review:

1. **Token-level DM detection** (no substring matches like "so" inside "also"), a principled DM inventory, and POS filters for ambiguous items.
2. **Normalized frequencies + significance**: rates per 10k words, log-likelihood (G²) at the corpus level, and Mann–Whitney tests at the *speaker* level, with Benjamini–Hochberg correction.
3. **Speaker-level train/val/test split**, so no speaker appears in more than one split.
4. **One shared, balanced test set** for every model.
5. **Ablations** that isolate DMs: `full` text, `dm_only` (just the DM sequence), `no_dm` (text with DMs removed).
6. Logistic regression and fine-tuned BERT on each condition, **best checkpoint by validation loss**, and **speaker-cluster bootstrap 95% CIs**, including paired differences between models.
7. A **speaker-level classifier** on DM rates (the cleanest test of "are DMs gendered?"), with a permutation test.
8. A **quantitative masked-LM probe** of pre-trained BERT (the old notebook loaded the fine-tuned classifier into a masked-LM head, which leaves that head randomly initialized).
9. Gender-encoding anonymization tags (`--ANONnameM` / `--ANONnameF`) are neutralized.

**To run:** Runtime → Change runtime type → **T4 GPU**. Put the Spoken BNC2014 *tagged* XML folder and `bnc2014spoken-speakerdata.tsv` in Google Drive, set the paths in the next cell, then Runtime → Run all.
Everything is written to `OUT_DIR`, and the last cell zips it as `results.zip`. Send that zip back to finish the paper.

Rough runtime on a T4 with the defaults: ~1–1.5 hours (mostly the three BERT runs).

In [ ]:
# ---------------- CONFIG: edit these ----------------
CORPUS_XML_DIR = "/content/drive/MyDrive/bnc2014/spoken/tagged"        # folder of tagged .xml files (searched recursively)
SPEAKER_TSV    = "/content/drive/MyDrive/bnc2014/bnc2014spoken-speakerdata.tsv"
OUT_DIR        = "/content/drive/MyDrive/dm_gender_v2_results"

MODEL_NAME = "bert-base-uncased"
SEED = 42

# Split proportions (by SPEAKER, separately within each gender)
SPLIT = {"train": 0.70, "val": 0.10, "test": 0.20}

# Training-set size for both models, per gender (None = use everything; much slower for BERT)
MAX_TRAIN_PER_CLASS = 50_000
MAX_VAL_PER_CLASS   = 5_000
MAX_TEST_PER_CLASS  = None       # None = the whole balanced test split
CAP_UTTS_PER_SPEAKER = None      # e.g. 2000 to stop a few very talkative speakers dominating

# BERT
RUN_BERT   = True
CONDITIONS = ["full", "no_dm", "dm_only"]
MAX_LEN    = 64
EPOCHS     = 3
LR         = 2e-5
BATCH      = 32

# Statistics
N_BOOT = 1000
MIN_WORDS_PER_SPEAKER = 1000     # speaker-level analyses only use speakers with at least this many words
N_PERMUTATIONS = 200
MIN_DM_TEST_N = 200              # error analysis: only DMs that occur in at least this many test utterances

In [ ]:
import os, sys, glob, json, math, re, random, time, zipfile
from collections import Counter, defaultdict
import xml.etree.ElementTree as ET

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and CORPUS_XML_DIR.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             roc_auc_score, confusion_matrix)
from sklearn.model_selection import StratifiedKFold, permutation_test_score, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures"); os.makedirs(FIG_DIR, exist_ok=True)
TAB_DIR = os.path.join(OUT_DIR, "tables");  os.makedirs(TAB_DIR, exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass
set_seed()

RESULTS = {"config": {k: v for k, v in globals().items()
                      if k.isupper() and isinstance(v, (int, float, str, list, dict, type(None)))}}
def save_results():
    with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("Output folder:", OUT_DIR)

## 1. Discourse-marker inventory

Grouped into five functional categories. Items from the first paper that are not discourse markers were removed: sequencers (*first, second, third*), modal verbs (*might, could be*), frequency/degree adverbs (*always, never, sometimes, almost, relatively, quite*), British intensifiers (*proper, dead*), and the answer words *yes/no*.

Ambiguous single words are filtered by their CLAWS7 POS tag: for example, *like* tagged as a verb (`VV*`) is not counted. Cell 3 prints the tag distribution for every item so these rules can be checked.

In [ ]:
DM_INVENTORY = {
    "filler":      ["uh", "um", "erm", "er"],
    "response":    ["yeah", "mm"],
    "interactional": ["oh", "well", "so", "like", "right", "okay", "anyway",
                      "you know", "i mean", "mind you", "you see"],
    "hedge":       ["sort of", "kind of", "maybe", "perhaps", "probably", "possibly",
                    "i guess", "i think", "fairly", "roughly", "pretty much", "in a way",
                    "more or less", "to some extent", "it seems", "as far as i can tell"],
    "booster":     ["really", "definitely", "absolutely", "totally", "certainly", "clearly",
                    "utterly", "indeed", "undoubtedly", "completely", "extremely", "entirely",
                    "surely", "literally", "for sure", "of course", "in fact",
                    "without a doubt", "to be honest", "as a matter of fact"],
}
# Exclude a match when the POS tag of its FIRST token starts with any of these prefixes
POS_EXCLUDE = {
    "like":    ["VV"],          # "I like it"
    "well":    ["JJ", "NN"],    # "I'm well", "a well"
    "right":   ["JJ", "NN"],    # "the right answer", "a right"
    "sort of": ["NN"],          # "what sort of car"
    "kind of": ["NN"],          # "that kind of thing"
    "so":      [],              # left unfiltered: DM "so" is tagged inconsistently (see paper)
}

DM_CATEGORY = {dm: cat for cat, items in DM_INVENTORY.items() for dm in items}
DM_LIST = list(DM_CATEGORY)
DM_TOKENS = sorted([(tuple(dm.split()), dm) for dm in DM_LIST], key=lambda x: -len(x[0]))  # longest first
MAX_DM_LEN = max(len(t) for t, _ in DM_TOKENS)
DM_BY_FIRST = defaultdict(list)
for toks, dm in DM_TOKENS:
    DM_BY_FIRST[toks[0]].append((toks, dm))
print(len(DM_LIST), "discourse markers in", len(DM_INVENTORY), "categories")

def find_dms(tokens, pos):
    """Return list of (start, end, dm) non-overlapping matches, longest match first."""
    out, i, n = [], 0, len(tokens)
    while i < n:
        hit = None
        for toks, dm in DM_BY_FIRST.get(tokens[i], ()):
            L = len(toks)
            if i + L <= n and tuple(tokens[i:i+L]) == toks:
                if any(pos[i].startswith(p) for p in POS_EXCLUDE.get(dm, [])):
                    continue
                hit = (i, i + L, dm); break
        if hit:
            out.append(hit); i = hit[1]
        else:
            i += 1
    return out

# quick self-test
_t = "oh well i also know that you know it was sort of fine".split()
_p = ["UH", "RR", "PPIS1", "RR", "VV0", "CST", "PPY", "VV0", "PPH1", "VBDZ", "RR21", "RR22", "JJ"]
assert [m[2] for m in find_dms(_t, _p)] == ["oh", "well", "you know", "sort of"], find_dms(_t, _p)

## 2. Parse the corpus

Each `<u>` (utterance) becomes one row with its speaker, the lowercased tokens and their POS tags. Anonymization and transcription tags such as `--ANONnameM` or `--UNCLEARWORD` are replaced with the neutral `[UNK]`, because `ANONnameM`/`ANONnameF` encode the gender of the person being named.

In [ ]:
def parse_corpus(xml_dir):
    files = sorted(glob.glob(os.path.join(xml_dir, "**", "*.xml"), recursive=True))
    assert files, f"No .xml files found under {xml_dir}"
    rows = []
    for k, fp in enumerate(files):
        root = ET.parse(fp).getroot()
        text_id = root.attrib.get("id", os.path.basename(fp))
        for u_idx, u in enumerate(root.iter("u")):
            toks, pos = [], []
            for w in u.iter("w"):
                t = (w.text or "").strip()
                if not t:
                    continue
                if t.startswith("--"):
                    t = "[UNK]"
                toks.append(t if t == "[UNK]" else t.lower())
                pos.append(w.attrib.get("pos", ""))
            if toks:
                rows.append({"text_id": text_id, "u_idx": u_idx,
                             "speaker": u.attrib.get("who", "UNK"),
                             "tokens": toks, "pos": pos})
        if (k + 1) % 200 == 0:
            print(f"  parsed {k+1}/{len(files)} files")
    print(f"Parsed {len(files)} files, {len(rows):,} utterances")
    return pd.DataFrame(rows)

t0 = time.time()
utts = parse_corpus(CORPUS_XML_DIR)
print(f"{time.time()-t0:.0f}s")

In [ ]:
def load_speaker_sex(tsv_path):
    """The speaker file has no header row, so find the columns: speaker id = values like S0001;
    sex = the column whose values are almost all M/F."""
    meta = pd.read_csv(tsv_path, sep="\t", header=None, dtype=str)
    id_col = max(meta.columns, key=lambda c: meta[c].fillna("").str.fullmatch(r"S\d{4}").mean())
    sex_col = max(meta.columns, key=lambda c: meta[c].fillna("").isin(["M", "F"]).mean())
    print(f"speaker id column = {id_col}, sex column = {sex_col} "
          f"({meta[sex_col].isin(['M','F']).mean():.0%} M/F)")
    m = meta[[id_col, sex_col]].rename(columns={id_col: "speaker", sex_col: "sex"})
    return m[m["sex"].isin(["M", "F"])].drop_duplicates("speaker")

meta = load_speaker_sex(SPEAKER_TSV)
n_before = len(utts)
unk_share = (~utts["speaker"].str.fullmatch(r"S\d{4}")).mean()
utts = utts.merge(meta, on="speaker", how="inner")   # drops UNKFEMALE/UNKMALE/UNKMULTI: not single speakers
utts["label"] = (utts["sex"] == "F").astype(int)     # 1 = female, 0 = male
print(f"Kept {len(utts):,}/{n_before:,} utterances; {unk_share:.1%} were from unidentified speakers (UNK*)")

In [ ]:
# DM matches, the three text conditions, and word counts
matches = [find_dms(t, p) for t, p in zip(utts["tokens"], utts["pos"])]
utts["dms"] = [[m[2] for m in ms] for ms in matches]
utts["n_words"] = utts["tokens"].str.len()
utts["n_dms"] = utts["dms"].str.len()

def remove_spans(tokens, ms):
    keep = np.ones(len(tokens), bool)
    for s, e, _ in ms:
        keep[s:e] = False
    return " ".join(t for t, k in zip(tokens, keep) if k)

utts["full"] = utts["tokens"].str.join(" ")
utts["dm_only"] = utts["dms"].str.join(" ")
utts["no_dm"] = [remove_spans(t, ms) for t, ms in zip(utts["tokens"], matches)]

# POS tag check for each DM's first token (inspect this for the POS rules above)
pos_check = defaultdict(Counter)
for toks, pos in zip(utts["tokens"], utts["pos"]):
    for i, t in enumerate(toks):
        for dtoks, dm in DM_BY_FIRST.get(t, ()):
            if tuple(toks[i:i+len(dtoks)]) == dtoks:
                pos_check[dm][pos[i]] += 1
pos_table = pd.DataFrame([{"dm": dm, "top_tags": ", ".join(f"{k}:{v}" for k, v in c.most_common(5)),
                           "excluded_by_rule": sum(v for k, v in c.items()
                                                   if any(k.startswith(p) for p in POS_EXCLUDE.get(dm, [])))}
                          for dm, c in pos_check.items()])
pos_table.to_csv(os.path.join(TAB_DIR, "dm_pos_check.csv"), index=False)
print(pos_table.to_string(index=False))

## 3. Corpus composition and normalized DM frequencies

Two levels of testing:
* **Corpus level:** log-likelihood (G²) on pooled counts, the standard corpus-linguistics test. It treats every word as independent, so it overstates significance.
* **Speaker level:** each speaker's rate per 1,000 words, compared between female and male speakers with a Mann–Whitney U test. This respects the fact that speakers differ, and it is the result to trust.

Both use Benjamini–Hochberg correction across the DM inventory.

In [ ]:
def bh(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    q = np.empty(n); prev = 1.0
    for rank, idx in reversed(list(enumerate(order, start=1))):
        prev = min(prev, p[idx] * n / rank); q[idx] = prev
    return q

words = utts.groupby("sex")["n_words"].sum()
spk_words = utts.groupby(["speaker", "sex"])["n_words"].sum().reset_index()
comp = {
    "n_speakers_F": int((spk_words.sex == "F").sum()), "n_speakers_M": int((spk_words.sex == "M").sum()),
    "words_F": int(words.get("F", 0)), "words_M": int(words.get("M", 0)),
    "utts_F": int((utts.sex == "F").sum()), "utts_M": int((utts.sex == "M").sum()),
    "share_utts_with_dm": float((utts.n_dms > 0).mean()),
    "share_utts_empty_without_dm": float(((utts.n_dms > 0) & (utts.no_dm.str.len() == 0)).sum() / max(1, (utts.n_dms > 0).sum())),
}
top10 = utts.groupby("speaker").size().sort_values(ascending=False)
comp["share_utts_top10_speakers"] = float(top10.head(10).sum() / top10.sum())
RESULTS["composition"] = comp
print(json.dumps(comp, indent=2))

# counts per DM per sex, and per speaker
dm_counts = Counter(); spk_dm = defaultdict(Counter)
for spk, sex, dms in zip(utts["speaker"], utts["sex"], utts["dms"]):
    for d in dms:
        dm_counts[(d, sex)] += 1; spk_dm[spk][d] += 1

spk_ok = spk_words[spk_words.n_words >= MIN_WORDS_PER_SPEAKER].set_index("speaker")
rate = pd.DataFrame({d: [1000 * spk_dm[s][d] / spk_ok.loc[s, "n_words"] for s in spk_ok.index] for d in DM_LIST},
                    index=spk_ok.index)
rate_sex = spk_ok["sex"]
RESULTS["composition"]["n_speakers_speaker_level"] = {"F": int((rate_sex == "F").sum()), "M": int((rate_sex == "M").sum())}

rows = []
wF, wM = words["F"], words["M"]
for d in DM_LIST:
    a, b = dm_counts[(d, "F")], dm_counts[(d, "M")]
    E1 = wF * (a + b) / (wF + wM); E2 = wM * (a + b) / (wF + wM)
    G2 = 2 * ((a * math.log(a / E1) if a else 0) + (b * math.log(b / E2) if b else 0))
    p_ll = stats.chi2.sf(G2, 1)
    log2r = math.log2(((a + .5) / wF) / ((b + .5) / wM))
    rF, rM = rate.loc[rate_sex == "F", d], rate.loc[rate_sex == "M", d]
    if (rF.sum() + rM.sum()) > 0:
        U, p_mw = stats.mannwhitneyu(rF, rM, alternative="two-sided")
        rbc = 2 * U / (len(rF) * len(rM)) - 1   # rank-biserial: >0 means higher in women
    else:
        p_mw, rbc = 1.0, 0.0
    rows.append({"dm": d, "category": DM_CATEGORY[d], "count_F": a, "count_M": b,
                 "per10k_F": 1e4 * a / wF, "per10k_M": 1e4 * b / wM, "log2_ratio_F_over_M": log2r,
                 "G2": G2, "p_LL": p_ll, "median_rate_F": rF.median(), "median_rate_M": rM.median(),
                 "p_MW": p_mw, "rank_biserial": rbc})
freq = pd.DataFrame(rows)
freq["q_LL"] = bh(freq["p_LL"]); freq["q_MW"] = bh(freq["p_MW"])
freq = freq.sort_values("log2_ratio_F_over_M")
freq.to_csv(os.path.join(TAB_DIR, "dm_frequencies.csv"), index=False)
RESULTS["frequencies"] = {
    "n_sig_LL": int((freq.q_LL < .05).sum()), "n_sig_MW": int((freq.q_MW < .05).sum()), "n_dms": len(freq),
    "n_sig_MW_higher_F": int(((freq.q_MW < .05) & (freq.rank_biserial > 0)).sum()),
    "n_sig_MW_higher_M": int(((freq.q_MW < .05) & (freq.rank_biserial < 0)).sum()),
    "max_abs_rank_biserial": float(freq.rank_biserial.abs().max()),
    "table": freq.round(6).to_dict(orient="records"),
}
# category totals
cat_rows = []
for cat, items in DM_INVENTORY.items():
    a = sum(dm_counts[(d, "F")] for d in items); b = sum(dm_counts[(d, "M")] for d in items)
    rF = rate.loc[rate_sex == "F", items].sum(1); rM = rate.loc[rate_sex == "M", items].sum(1)
    U, p = stats.mannwhitneyu(rF, rM, alternative="two-sided")
    cat_rows.append({"category": cat, "per10k_F": 1e4 * a / wF, "per10k_M": 1e4 * b / wM,
                     "log2_ratio_F_over_M": math.log2(((a + .5) / wF) / ((b + .5) / wM)),
                     "p_MW": p, "rank_biserial": 2 * U / (len(rF) * len(rM)) - 1})
cats = pd.DataFrame(cat_rows); cats["q_MW"] = bh(cats["p_MW"])
cats.to_csv(os.path.join(TAB_DIR, "category_frequencies.csv"), index=False)
RESULTS["categories"] = cats.round(6).to_dict(orient="records")
print(cats.round(4).to_string(index=False))
print(freq[["dm", "category", "per10k_F", "per10k_M", "log2_ratio_F_over_M", "q_LL", "q_MW", "rank_biserial"]].round(4).to_string(index=False))

In [ ]:
# Figure: log ratio per DM (only DMs with >= 50 occurrences), marking speaker-level significance
CAT_COLORS = {"filler": "#7f7f7f", "response": "#17becf", "interactional": "#1f77b4", "hedge": "#2ca02c", "booster": "#d62728"}
fplot = freq[(freq.count_F + freq.count_M) >= 50].sort_values("log2_ratio_F_over_M")
fig, ax = plt.subplots(figsize=(5.2, 0.17 * len(fplot) + 1.2))
ax.barh(fplot.dm, fplot.log2_ratio_F_over_M, color=[CAT_COLORS[c] for c in fplot.category])
for y, (_, r) in enumerate(fplot.iterrows()):
    if r.q_MW < .05:
        x = r.log2_ratio_F_over_M
        ax.text(x + (0.03 if x >= 0 else -0.03), y, "*", va="center", ha="left" if x >= 0 else "right", fontsize=9)
ax.axvline(0, color="k", lw=.8)
ax.set_xlabel("log$_2$ ratio of rate per word (women / men)")
ax.tick_params(axis="y", labelsize=7)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CAT_COLORS.values()]
ax.legend(handles, CAT_COLORS.keys(), fontsize=7, loc="lower right", frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "dm_log_ratio.pdf")); fig.savefig(os.path.join(FIG_DIR, "dm_log_ratio.png"), dpi=200)
plt.close(fig)

## 4. Speaker-level classifier: can a speaker's DM rates predict their gender?

One row per speaker (≥ `MIN_WORDS_PER_SPEAKER` words) with their rate of every DM per 1,000 words. Repeated stratified 5-fold cross-validation, plus a permutation test (labels shuffled `N_PERMUTATIONS` times) to show whether accuracy beats chance.

In [ ]:
Xs = rate.values; ys = (rate_sex == "F").astype(int).values
clf_spk = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
accs = []
for rep in range(5):
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED + rep)
    accs += list(cross_val_score(clf_spk, Xs, ys, cv=cv, scoring="balanced_accuracy"))
score, perm_scores, p_perm = permutation_test_score(
    clf_spk, Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    n_permutations=N_PERMUTATIONS, scoring="balanced_accuracy", random_state=SEED)
clf_spk.fit(Xs, ys)
coefs = pd.Series(clf_spk[-1].coef_[0], index=DM_LIST).sort_values()
RESULTS["speaker_level"] = {"n_speakers": int(len(ys)), "balanced_acc_mean": float(np.mean(accs)),
                            "balanced_acc_sd": float(np.std(accs)), "perm_p": float(p_perm),
                            "perm_null_mean": float(np.mean(perm_scores)),
                            "top_female_coefs": coefs.tail(5).round(3).to_dict(),
                            "top_male_coefs": coefs.head(5).round(3).to_dict()}
print(json.dumps(RESULTS["speaker_level"], indent=2)); save_results()

## 5. Utterance-level data: speaker split, balancing, shared test set

Only utterances containing at least one DM are used, as in the original study. Speakers are assigned to train/val/test **separately within each gender**, so every split has speakers of both genders and no speaker appears twice. Each split is then balanced by downsampling the larger gender. **Every model is evaluated on exactly the same test rows.**

In [ ]:
rng = np.random.RandomState(SEED)
data = utts[utts.n_dms > 0].copy()
if CAP_UTTS_PER_SPEAKER:
    data = pd.concat([g.sample(min(len(g), CAP_UTTS_PER_SPEAKER), random_state=SEED)
                      for _, g in data.groupby("speaker")])

split_of = {}
for sex in ["F", "M"]:
    spk = sorted(data.loc[data.sex == sex, "speaker"].unique()); rng.shuffle(spk)
    n = len(spk); a = int(round(SPLIT["train"] * n)); b = a + int(round(SPLIT["val"] * n))
    for i, s in enumerate(spk):
        split_of[s] = "train" if i < a else ("val" if i < b else "test")
data["split"] = data.speaker.map(split_of)

def balance(df, cap):
    n = min(df.label.value_counts().min(), cap or 10**12)
    parts = [df[df.label == lab].sample(n, random_state=SEED) for lab in (0, 1)]
    return pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

train = balance(data[data.split == "train"], MAX_TRAIN_PER_CLASS)
val   = balance(data[data.split == "val"],   MAX_VAL_PER_CLASS)
test  = balance(data[data.split == "test"],  MAX_TEST_PER_CLASS)
assert not (set(train.speaker) & set(test.speaker)) and not (set(val.speaker) & set(test.speaker))
RESULTS["splits"] = {name: {"utterances": int(len(d)), "speakers": int(d.speaker.nunique()),
                            "speakers_F": int(d[d.label == 1].speaker.nunique()),
                            "speakers_M": int(d[d.label == 0].speaker.nunique())}
                     for name, d in [("train", train), ("val", val), ("test", test)]}
lens = train["full"].str.split().str.len()
RESULTS["splits"]["median_words_per_utt"] = float(lens.median())
RESULTS["splits"]["p95_words_per_utt"] = float(lens.quantile(.95))
print(json.dumps(RESULTS["splits"], indent=2)); save_results()
test[["text_id", "u_idx", "speaker", "label"]].to_csv(os.path.join(TAB_DIR, "test_set_ids.csv"), index=False)

## 6. Evaluation helpers: metrics and speaker-cluster bootstrap

Utterances from the same speaker are not independent, so confidence intervals resample **test speakers**, not utterances. Paired differences between two models reuse the same resampled speakers.

In [ ]:
y_test = test.label.values
test_spk = test.speaker.values
_spk_index = {s: np.where(test_spk == s)[0] for s in np.unique(test_spk)}
_spk_by_sex = {lab: [s for s in _spk_index if test.label.values[_spk_index[s][0]] == lab] for lab in (0, 1)}
_boot_rng = np.random.RandomState(SEED)
BOOT_SAMPLES = []
for _ in range(N_BOOT):   # stratified by gender so every replicate has both classes
    idx = np.concatenate([_spk_index[s] for lab in (0, 1)
                          for s in _boot_rng.choice(_spk_by_sex[lab], len(_spk_by_sex[lab]), replace=True)])
    BOOT_SAMPLES.append(idx)

def metrics(y, prob):
    pred = (prob >= .5).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=[0, 1], zero_division=0)
    out = {"acc": accuracy_score(y, pred), "macro_f1": f1_score(y, pred, average="macro"),
           "auc": roc_auc_score(y, prob) if len(set(y)) > 1 else float("nan"),
           "P_M": p[0], "R_M": r[0], "F1_M": f[0], "P_F": p[1], "R_F": r[1], "F1_F": f[1],
           "pred_F_share": float(pred.mean()), "confusion": confusion_matrix(y, pred, labels=[0, 1]).tolist()}
    return out

def boot_ci(prob, key="acc", prob_b=None):
    vals = []
    for idx in BOOT_SAMPLES:
        y = y_test[idx]
        f = lambda pr: accuracy_score(y, pr[idx] >= .5) if key == "acc" else f1_score(y, pr[idx] >= .5, average="macro")
        vals.append(f(prob) - (f(prob_b) if prob_b is not None else 0))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return float(lo), float(hi)

def speaker_level_acc(prob):
    df = pd.DataFrame({"spk": test_spk, "y": y_test, "p": prob}).groupby("spk").agg(y=("y", "first"), p=("p", "mean"))
    return float(balanced_acc(df.y.values, (df.p >= .5).astype(int).values)), int(len(df))

def balanced_acc(y, pred):
    return np.mean([np.mean(pred[y == c] == c) for c in (0, 1)])

PROBS = {}   # model name -> test probabilities of "female"
EVAL = {}
def record(name, prob):
    PROBS[name] = prob
    m = metrics(y_test, prob)
    m["acc_ci"] = boot_ci(prob, "acc"); m["macro_f1_ci"] = boot_ci(prob, "f1")
    m["speaker_bal_acc"], m["n_test_speakers"] = speaker_level_acc(prob)
    EVAL[name] = m
    RESULTS["eval"] = EVAL; save_results()
    print(f"{name:>14}: acc={m['acc']:.3f} [{m['acc_ci'][0]:.3f}, {m['acc_ci'][1]:.3f}]  "
          f"macroF1={m['macro_f1']:.3f}  AUC={m['auc']:.3f}  R_M={m['R_M']:.2f} R_F={m['R_F']:.2f}  "
          f"speaker-level={m['speaker_bal_acc']:.3f}")

record("chance", np.full(len(y_test), .5) + np.random.RandomState(SEED).uniform(-1e-6, 1e-6, len(y_test)))

## 7. Logistic regression (unigram + bigram counts) on each condition

In [ ]:
LR_TOP = {}
for cond in CONDITIONS:
    vec = CountVectorizer(ngram_range=(1, 2), min_df=2, token_pattern=r"(?u)\b\w+\b|\[unk\]", lowercase=True)
    Xtr = vec.fit_transform(train[cond]); Xte = vec.transform(test[cond])
    lr = LogisticRegression(C=1.0, max_iter=3000)
    lr.fit(Xtr, train.label)
    record(f"LR_{cond}", lr.predict_proba(Xte)[:, 1])
    names = np.array(vec.get_feature_names_out()); co = lr.coef_[0]; o = np.argsort(co)
    LR_TOP[cond] = {"female": names[o[-15:]][::-1].tolist(), "male": names[o[:15]].tolist()}
RESULTS["lr_top_features"] = LR_TOP; save_results()
print(json.dumps(LR_TOP, indent=1))

## 8. Fine-tuned BERT on each condition

Same training rows as logistic regression. Evaluated on the validation set after each epoch; the checkpoint with the **lowest validation loss** is kept and used on the test set.

In [ ]:
if RUN_BERT:
    import torch
    from torch.utils.data import Dataset
    from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                              TrainingArguments, DataCollatorWithPadding)
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be very slow)")
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)

    class TextDS(Dataset):
        def __init__(self, texts, labels):
            self.enc = tok(list(texts), truncation=True, max_length=MAX_LEN)
            self.labels = list(labels)
        def __len__(self): return len(self.labels)
        def __getitem__(self, i):
            item = {k: v[i] for k, v in self.enc.items()}; item["labels"] = int(self.labels[i]); return item

    TRAIN_LOGS = {}
    for cond in CONDITIONS:
        set_seed()
        model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        args = TrainingArguments(
            output_dir=f"/tmp/bert_{cond}", eval_strategy="epoch", save_strategy="epoch",
            learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=256,
            num_train_epochs=EPOCHS, weight_decay=0.01,
            warmup_steps=int(0.06 * EPOCHS * math.ceil(len(train) / BATCH)),  # 6% linear warm-up
            load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
            save_total_limit=1, fp16=torch.cuda.is_available(), report_to="none", seed=SEED,
            logging_strategy="epoch", dataloader_num_workers=2)
        trainer = Trainer(model=model, args=args,
                          train_dataset=TextDS(train[cond], train.label),
                          eval_dataset=TextDS(val[cond], val.label),
                          data_collator=DataCollatorWithPadding(tok))
        t0 = time.time(); trainer.train()
        logs = [{"epoch": h["epoch"], **{k: h[k] for k in ("loss", "eval_loss") if k in h}}
                for h in trainer.state.log_history if "loss" in h or "eval_loss" in h]
        best_ckpt = trainer.state.best_model_checkpoint
        TRAIN_LOGS[cond] = {"log": logs, "best_checkpoint": best_ckpt,
                            "best_eval_loss": trainer.state.best_metric, "minutes": (time.time() - t0) / 60}
        logits = trainer.predict(TextDS(test[cond], test.label)).predictions
        prob = torch.softmax(torch.tensor(logits, dtype=torch.float32), -1)[:, 1].numpy()
        record(f"BERT_{cond}", prob)
        RESULTS["bert_training"] = TRAIN_LOGS; save_results()
        del model, trainer
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # training curves
    fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(3 * len(CONDITIONS), 2.4), sharey=True)
    for ax, cond in zip(np.atleast_1d(axes), CONDITIONS):
        lg = TRAIN_LOGS[cond]["log"]
        tr = [(l["epoch"], l["loss"]) for l in lg if "loss" in l]; ev = [(l["epoch"], l["eval_loss"]) for l in lg if "eval_loss" in l]
        if tr: ax.plot(*zip(*tr), "o-", label="train")
        if ev: ax.plot(*zip(*ev), "s-", label="validation")
        ax.set_title(cond, fontsize=9); ax.set_xlabel("epoch")
    np.atleast_1d(axes)[0].set_ylabel("cross-entropy loss"); np.atleast_1d(axes)[0].legend(fontsize=7)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "bert_training_curves.pdf")); plt.close(fig)

## 9. Paired comparisons (speaker-cluster bootstrap)

A difference is reliable when its 95% CI excludes 0.

In [ ]:
pairs = [("LR_full", "chance"), ("LR_dm_only", "chance"), ("LR_full", "LR_no_dm"), ("LR_full", "LR_dm_only")]
if RUN_BERT:
    pairs += [("BERT_full", "LR_full"), ("BERT_dm_only", "LR_dm_only"), ("BERT_no_dm", "LR_no_dm"),
              ("BERT_full", "BERT_no_dm"), ("BERT_full", "BERT_dm_only"), ("BERT_dm_only", "chance")]
COMP = {}
for a, b in pairs:
    if a in PROBS and b in PROBS:
        d = EVAL[a]["acc"] - EVAL[b]["acc"]; lo, hi = boot_ci(PROBS[a], "acc", PROBS[b])
        COMP[f"{a}_vs_{b}"] = {"diff_acc": d, "ci": [lo, hi], "reliable": bool(lo > 0 or hi < 0)}
        print(f"{a:>13} - {b:<12} acc diff = {d:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
RESULTS["comparisons"] = COMP; save_results()

## 10. Error analysis: accuracy on utterances containing each DM

In [ ]:
err_rows = []
best = "BERT_full" if "BERT_full" in PROBS else "LR_full"
pred_best = (PROBS[best] >= .5).astype(int)
for d in DM_LIST:
    mask = test.dms.apply(lambda l: d in l).values
    if mask.sum() >= MIN_DM_TEST_N:
        err_rows.append({"dm": d, "n": int(mask.sum()), "acc": float((pred_best[mask] == y_test[mask]).mean()),
                         "pred_F_share": float(pred_best[mask].mean()), "true_F_share": float(y_test[mask].mean())})
err = pd.DataFrame(err_rows, columns=["dm", "n", "acc", "pred_F_share", "true_F_share"]).sort_values("pred_F_share")
err.to_csv(os.path.join(TAB_DIR, f"per_dm_predictions_{best}.csv"), index=False)
RESULTS["per_dm_predictions"] = {"model": best, "table": err.round(4).to_dict(orient="records")}

lens_t = test["full"].str.split().str.len().values
bins = [0, 3, 6, 11, 21, 10**6]; labels_b = ["1-2", "3-5", "6-10", "11-20", "21+"]
by_len = {}
for name in [k for k in PROBS if k != "chance"]:
    pr = (PROBS[name] >= .5).astype(int)
    by_len[name] = {lab: float((pr[(lens_t >= lo) & (lens_t < hi)] == y_test[(lens_t >= lo) & (lens_t < hi)]).mean())
                    for lab, lo, hi in zip(labels_b, bins[:-1], bins[1:]) if ((lens_t >= lo) & (lens_t < hi)).any()}
RESULTS["acc_by_length"] = by_len
mis = test.assign(prob_F=PROBS[best], pred=pred_best)[lambda d: d.pred != d.label]
mis[["speaker", "sex", "full", "dm_only", "prob_F"]].sample(min(300, len(mis)), random_state=SEED) \
    .to_csv(os.path.join(TAB_DIR, f"misclassified_sample_{best}.csv"), index=False)
save_results()
print(err.round(3).to_string(index=False)); print(json.dumps(by_len, indent=1))

## 11. Masked-LM probe of pre-trained BERT

Does the pre-trained (not fine-tuned) model associate a DM with a gender? For each template, BERT fills `[MASK]` and we compare the probability of male words with female words:

score = log( P(male words) / P(female words) ), and the DM's **shift** = score with the DM − score of the same template without it.

A negative shift means the DM moves BERT toward female words. Shifts are then correlated (Spearman) with the corpus log ratios from Section 3.

In [ ]:
if RUN_BERT:
    from transformers import AutoModelForMaskedLM
    mlm_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; mlm.to(dev)
    TEMPLATES = [
        ("{dm}, [MASK] said it was fine.", ["he"], ["she"]),
        ("{dm}, [MASK] was there yesterday.", ["he"], ["she"]),
        ("{dm}, [MASK] thinks it's a good idea.", ["he"], ["she"]),
        ("{dm}, [MASK] didn't want to go.", ["he"], ["she"]),
        ("{dm}, I'm just a [MASK].", ["man", "guy", "boy", "lad", "bloke"], ["woman", "girl", "lady", "lass"]),
        ("{dm}, that [MASK] over there told me.", ["man", "guy", "boy", "lad", "bloke"], ["woman", "girl", "lady", "lass"]),
        ("{dm}, my [MASK] is coming round later.", ["husband", "boyfriend", "dad", "brother", "son"],
                                                     ["wife", "girlfriend", "mum", "sister", "daughter"]),
    ]
    def mask_score(sentence, male, female):
        enc = mlm_tok(sentence, return_tensors="pt").to(dev)
        pos_m = (enc.input_ids[0] == mlm_tok.mask_token_id).nonzero()[0].item()
        with torch.no_grad():
            p = torch.softmax(mlm(**enc).logits[0, pos_m], -1)
        ids = lambda ws: [VOCAB[w] for w in ws if w in VOCAB]
        return float(torch.log(p[ids(male)].sum() / p[ids(female)].sum()))
    def strip(t):  # template without the DM
        s = t.replace("{dm}, ", "").replace("{dm} ", "")
        return s[0].upper() + s[1:]
    VOCAB = mlm_tok.get_vocab()
    for t_, m_, f_ in TEMPLATES:
        assert any(w in VOCAB for w in m_) and any(w in VOCAB for w in f_), f"no gender words in vocab for: {t_}"
    print("Gender words missing from the vocabulary (skipped):",
          sorted({w for _, m_, f_ in TEMPLATES for w in m_ + f_ if w not in VOCAB}))
    base = [mask_score(strip(t), m, f) for t, m, f in TEMPLATES]
    probe_rows = []
    for d in DM_LIST:
        shifts = [mask_score(t.format(dm=d.capitalize() if not d.startswith("i ") else "I" + d[1:]), m, f) - b0
                  for (t, m, f), b0 in zip(TEMPLATES, base)]
        probe_rows.append({"dm": d, "category": DM_CATEGORY[d], "mean_shift": float(np.mean(shifts)),
                           "sd_shift": float(np.std(shifts))})
    probe = pd.DataFrame(probe_rows).merge(freq[["dm", "log2_ratio_F_over_M", "count_F", "count_M"]], on="dm")
    probe.to_csv(os.path.join(TAB_DIR, "mlm_probe.csv"), index=False)
    ok = probe[(probe.count_F + probe.count_M) >= 50]
    rho, p_rho = stats.spearmanr(-ok.mean_shift, ok.log2_ratio_F_over_M)  # sign flipped so + = both lean female
    RESULTS["mlm_probe"] = {"baseline_scores": base, "spearman_rho": float(rho), "spearman_p": float(p_rho),
                            "n": int(len(ok)), "table": probe.round(4).to_dict(orient="records")}
    save_results()
    print(probe.sort_values("mean_shift").round(3).to_string(index=False))
    print(f"Spearman rho (BERT female-lean vs corpus female-lean) = {rho:.3f}, p = {p_rho:.3g}, n = {len(ok)}")

## 12. Export LaTeX tables and macros, then zip everything

`results_macros.tex` and the `tables/*.tex` files are read directly by the paper.

In [ ]:
def f3(x): return f"{x:.3f}"
def macro_name(s): return re.sub(r"[^A-Za-z]", "", s.title())
mac = []
def M(name, val): mac.append(f"\\newcommand{{\\{name}}}{{{val}}}")
c = RESULTS["composition"]
M("NSpkF", c["n_speakers_F"]); M("NSpkM", c["n_speakers_M"])
M("WordsF", f"{c['words_F']:,}"); M("WordsM", f"{c['words_M']:,}")
M("ShareUttsDM", f"{100*c['share_utts_with_dm']:.1f}\\%")
M("ShareTopTen", f"{100*c['share_utts_top10_speakers']:.1f}\\%")
fr = RESULTS["frequencies"]
M("NDMs", fr["n_dms"]); M("NSigLL", fr["n_sig_LL"]); M("NSigMW", fr["n_sig_MW"])
M("NSigMWF", fr["n_sig_MW_higher_F"]); M("NSigMWM", fr["n_sig_MW_higher_M"]); M("MaxRBC", f"{fr['max_abs_rank_biserial']:.2f}")
sl = RESULTS["speaker_level"]
M("SpkN", sl["n_speakers"]); M("SpkAcc", f3(sl["balanced_acc_mean"])); M("SpkAccSD", f3(sl["balanced_acc_sd"]))
M("SpkPermP", f"{sl['perm_p']:.3f}")
for sp in ("train", "val", "test"):
    M(f"N{sp.title()}Utts", f"{RESULTS['splits'][sp]['utterances']:,}"); M(f"N{sp.title()}Spk", RESULTS["splits"][sp]["speakers"])
for name, m in EVAL.items():
    k = macro_name(name)
    M(f"Acc{k}", f3(m["acc"])); M(f"AccLo{k}", f3(m["acc_ci"][0])); M(f"AccHi{k}", f3(m["acc_ci"][1]))
    M(f"FOne{k}", f3(m["macro_f1"])); M(f"AUC{k}", f3(m["auc"])); M(f"RM{k}", f"{m['R_M']:.2f}"); M(f"RF{k}", f"{m['R_F']:.2f}")
    M(f"SpkAcc{k}", f3(m["speaker_bal_acc"]))
for key, v in COMP.items():
    k = macro_name(key)
    M(f"Diff{k}", f"{100*v['diff_acc']:+.1f}"); M(f"DiffLo{k}", f"{100*v['ci'][0]:+.1f}"); M(f"DiffHi{k}", f"{100*v['ci'][1]:+.1f}")
if "mlm_probe" in RESULTS:
    M("ProbeRho", f"{RESULTS['mlm_probe']['spearman_rho']:.2f}"); M("ProbeP", f"{RESULTS['mlm_probe']['spearman_p']:.3f}")
    M("ProbeN", RESULTS["mlm_probe"]["n"])
with open(os.path.join(OUT_DIR, "results_macros.tex"), "w") as f:
    f.write("% Generated by the v2 notebook. Do not edit by hand.\n" + "\n".join(mac) + "\n")

# Table: main results
order = ["chance"] + [f"{m}_{c}" for m in ("LR", "BERT") for c in CONDITIONS]
lines = [r"\begin{tabular}{llcccc}", r"\toprule",
         r"Model & Input & Accuracy [95\% CI] & Macro-F1 & AUC & Speaker acc. \\"]
COND_LABEL = {"full": "full text", "no_dm": "DMs removed", "dm_only": "DMs only"}
for group in (["chance"], [f"LR_{c}" for c in CONDITIONS], [f"BERT_{c}" for c in CONDITIONS]):
    rows_g = [g for g in group if g in EVAL]
    if not rows_g: continue
    lines.append(r"\midrule")
    for name in rows_g:
        m = EVAL[name]
        mod, cond = ("Chance", "--") if name == "chance" else (name.split("_", 1)[0], COND_LABEL[name.split("_", 1)[1]])
        lines.append(f"{mod} & {cond} & {m['acc']:.3f} [{m['acc_ci'][0]:.3f}, {m['acc_ci'][1]:.3f}] "
                     f"& {m['macro_f1']:.3f} & {m['auc']:.3f} & {m['speaker_bal_acc']:.3f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, "results_table.tex"), "w").write("\n".join(lines))

# Table: DM frequencies (DMs with >= 200 occurrences, sorted by log ratio)
ft = freq[(freq.count_F + freq.count_M) >= 200].sort_values("log2_ratio_F_over_M", ascending=False)
def star(q): return "$^{***}$" if q < .001 else "$^{**}$" if q < .01 else "$^{*}$" if q < .05 else ""
lines = [r"\begin{tabular}{llrrrr}", r"\toprule",
         r"DM & Category & Women & Men & log$_2$ ratio & $r_{rb}$ \\",
         r" & & \multicolumn{2}{c}{per 10k words} & & \\", r"\midrule"]
for _, r in ft.iterrows():
    lines.append(f"\\textit{{{r.dm}}} & {r.category} & {r.per10k_F:.1f} & {r.per10k_M:.1f} & "
                 f"{r.log2_ratio_F_over_M:+.2f}{star(r.q_LL)} & {r.rank_biserial:+.2f}{star(r.q_MW)} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(TAB_DIR, "dm_table.tex"), "w").write("\n".join(lines))
save_results()

zpath = os.path.join(OUT_DIR, "results.zip")
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for root_, _, fs in os.walk(OUT_DIR):
        for fn in fs:
            if fn != "results.zip":
                full = os.path.join(root_, fn); z.write(full, os.path.relpath(full, OUT_DIR))
print("Done. Send this file back:", zpath)
if IN_COLAB:
    from google.colab import files
    files.download(zpath)